# Week 7：XGBoost 调参——树深度

目标：固定其余参数，只改变 `max_depth`，用交叉验证观察模型复杂度如何影响训练集与验证集表现。

本节结论不是记住某个最佳数字，而是学会读取两条信号：训练平均分与验证平均分。

## 先建立判断规则

- 训练分数低、验证分数也低：模型可能太简单，称为 **underfitting（欠拟合）**。
- 训练分数很高、验证分数明显更低：模型可能记住了训练数据，称为 **overfitting（过拟合）**。
- 验证平均分高且验证标准差较小：在当前候选中更值得优先考虑。

这里的 `max_depth` 越大，每棵树可表达的规则越复杂。

In [1]:
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from xgboost import XGBClassifier

In [2]:
dataset = load_breast_cancer(as_frame=True)
X = dataset.data
y = dataset.target

# X_test 保留到模型选择全部完成之后，本实验只在 X_train 上做交叉验证。
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [3]:
rows = []

for depth in [1, 2, 3, 4, 5]:
    model = XGBClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=depth,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        eval_metric='logloss',
        random_state=42,
        n_jobs=1,
    )
    scores = cross_validate(
        model, X_train, y_train, cv=cv, scoring='accuracy',
        return_train_score=True, n_jobs=-1,
    )
    rows.append({
        'max_depth': depth,
        'train_mean': scores['train_score'].mean(),
        'validation_mean': scores['test_score'].mean(),
        'validation_std': scores['test_score'].std(),
    })

depth_results = pd.DataFrame(rows)
depth_results['gap'] = depth_results['train_mean'] - depth_results['validation_mean']
display(depth_results.sort_values('max_depth'))

,max_depth,train_mean,validation_mean,validation_std,gap
0,1,0.990110,0.962637,0.016447,0.027473
1,2,0.998352,0.971429,0.014906,0.026923
2,3,1.000000,0.973626,0.016447,0.026374
3,4,1.000000,0.967033,0.013900,0.032967
4,5,1.000000,0.969231,0.014579,0.030769


## 怎么读表

`gap` 是训练平均准确率减去验证平均准确率；它越大，越要警惕过拟合，但不能只凭它选模型。选择时先看验证平均分，再看标准差和 gap。

思考题：如果深度 5 的训练分数高于深度 2，但验证分数反而低，你会选择哪个？请用“泛化”解释原因。